In [1]:
import pandas  as pd
import numpy as np
import pdb, os, datetime, itertools, time, hashlib
import pyprojroot, sys
from pyprojroot.criterion import has_file
sys.path.insert(0, str(pyprojroot.find_root(has_file("pyproject.toml"))))
from dotenv import load_dotenv

load_dotenv()
from lib.flp001 import *

/workspace/worker/pj/Chrono/genuis/mizar
/workspace/worker/pj/Chrono/genuis/mizar/config/contract.toml


In [2]:
method = 'ricso2'
task_id = '113001'
session = '20260325'
instruments = 'rbb'
period = 5
category = 1

In [3]:
results = load_data(method=method, task_id=task_id, instruments=instruments, 
          period=period, session=session, category=category)
results['name'] = results['name'].astype(int).astype(int)
results['abs_ic'] = np.abs(results['ic_mean'])
#results['abs_icir'] = np.abs(results['icir'])
results1 = results.dropna(subset=['ann_sharpe','calmar'])
results1[['abs_ic','ann_sharpe','calmar']].tail()

/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260325


,abs_ic,ann_sharpe,calmar
95,0.0905,2.60,3.74
96,0.1430,2.67,4.12
97,0.0917,1.90,1.73
98,0.0780,2.24,2.98
99,0.0493,2.55,3.87


In [4]:
# results[results.expression.isin(drafit_data.formula.tolist())].to_csv("rbb_drafit.csv")
results1['abs_ic'].median()

0.1149

In [5]:
### IC大于0.03  ## 夏普大于2.2  # 卡玛大于2.5
results = results[(results['abs_ic']>0.03) & (results['ann_sharpe']>2) & (
    results['calmar']>2.2)].reset_index(drop=True)

In [6]:
results

,name,expression,avg_ret,total_ret,sharpe,ann_sharpe,max_dd,calmar,win_rate,pl_ratio,ic_mean,icir,turnover,factor_ac,ret_ac,roll_win,resampling_win,holding_profit,category,abs_ic
0,10115252,"MRes(240,MMinDiff(120,'tc004_10_10_15_1'),MDIF...",0.23,9581.99,0.03,3.05,-8.44,4.85,None,None,-0.1542,-0.6413,0.4297,0.5574,-0.0577,15.0,5.0,nxt1_ret_5h,p,0.1542
1,10254483,"EMA(5,EMA(5,MADiff(90,'tv007_1_2_1')))",0.20,5374.72,0.02,2.58,-10.14,3.46,None,None,-0.1244,-0.4962,0.4362,0.5042,-0.0594,15.0,5.0,nxt1_ret_5h,p,0.1244
2,10568718,"MRes(240,MA(240,'tc004_10_10_15_1'),'oi004_5_1...",0.23,8641.32,0.03,3.01,-7.60,5.25,None,None,-0.1618,-0.6792,0.4252,0.6030,-0.0574,15.0,5.0,nxt1_ret_5h,p,0.1618
3,10666782,"MCPS(10,MQUANTILE(60,MDPO(90,'low')))",0.16,2254.98,0.02,2.27,-9.22,2.90,None,None,0.0379,0.1411,0.4237,0.2338,-0.0381,15.0,5.0,nxt1_ret_5h,p,0.0379
4,10676140,"MMedian(5,'oi004_1_2_1')",0.20,5009.78,0.03,2.93,-8.08,4.25,None,None,-0.1012,-0.3998,0.4425,0.0626,-0.0086,15.0,5.0,nxt1_ret_5h,p,0.1012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,10094815,SIGLOG2ABS('oi034_5_10_1'),0.21,5969.99,0.03,2.78,-12.48,2.89,None,None,-0.1173,-0.4750,0.4058,0.4263,-0.0594,15.0,5.0,nxt1_ret_5h,p,0.1173
79,10164328,"MDEMA(15,MDPO(90,'oi039_2_3_1'))",0.20,5030.53,0.02,2.60,-9.20,3.74,None,None,0.0905,0.3635,0.4408,0.4344,-0.0586,15.0,5.0,nxt1_ret_5h,p,0.0905
80,10851056,"MA(10,'iv012_1_2_1')",0.21,5882.93,0.03,2.67,-8.72,4.12,None,None,0.1430,0.6026,0.4230,0.5854,-0.0380,15.0,5.0,nxt1_ret_5h,p,0.1430
81,10273729,"MADiff(30,WMA(5,'oi004_5_10_1'))",0.19,3711.30,0.02,2.24,-10.54,2.98,None,None,-0.0780,-0.3138,0.4211,0.5765,-0.0387,15.0,5.0,nxt1_ret_5h,p,0.0780


In [7]:
results = results[['expression','name']]
session_name = "/{0}".format(session)  if category == 1 else "/d{0}".format(session)  
results['plot'] = (base_path + '/' + str(method) + '/' + instruments + '/rulex' + '/' + str(task_id) + "/nxt1_ret_{0}h".format(period) \
    + session_name + '/plot/' + + results['name'].astype(str) + '.png')
results = results[['expression','plot','name']].rename(columns={'name':'factor_id','expression':'formula'})
results['plot'] = results['plot'].apply(make_clickable)

In [8]:
results['plot'].head().loc[0]

'<a target="_blank" href="/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260325/plot/10115252.png">/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260325/plot/10115252.png</a>'

In [9]:
### IC大于 total ic 0.02
### MA: total ic 0.015 mean ic 0.02  夏普 1.7 卡玛2.2
to_html(results)

url,formula,plot,factor_id
10115252,"MRes(240,MMinDiff(120,'tc004_10_10_15_1'),MDIFF(30,'oi004_5_10_1'))",/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260325/plot/10115252.png,10115252
10254483,"EMA(5,EMA(5,MADiff(90,'tv007_1_2_1')))",/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260325/plot/10254483.png,10254483
10568718,"MRes(240,MA(240,'tc004_10_10_15_1'),'oi004_5_10_1')",/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260325/plot/10568718.png,10568718
10666782,"MCPS(10,MQUANTILE(60,MDPO(90,'low')))",/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260325/plot/10666782.png,10666782
10676140,"MMedian(5,'oi004_1_2_1')",/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260325/plot/10676140.png,10676140
10497678,"WMA(5,EMA(5,MADiff(90,'tv007_2_3_1')))",/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260325/plot/10497678.png,10497678
10893906,"MQUANTILE(240,MDIFF(90,'oi004_5_10_1'))",/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260325/plot/10893906.png,10893906
10733688,"ASIN(ASIN(MDIFF(5,'oi004_5_10_1')))",/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260325/plot/10733688.png,10733688
10335500,"MDIFF(5,MDIFF(5,MDPO(60,MDIFF(5,'oi004_5_10_1'))))",/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260325/plot/10335500.png,10335500
10451725,"EMA(5,'tv007_1_2_1')",/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260325/plot/10451725.png,10451725


In [10]:
results1.sort_values(by=['icir'],ascending=False).max()

name                                                    10980146
expression        TANH(MQUANTILE(120,SIGLOG2ABS('dv002_2_3_1')))
avg_ret                                                     0.17
total_ret                                                1805.23
sharpe                                                      0.02
ann_sharpe                                                  2.31
max_dd                                                      -9.6
calmar                                                      2.64
win_rate                                                    None
pl_ratio                                                    None
ic_mean                                                   0.2331
icir                                                      0.9616
turnover                                                  0.4343
factor_ac                                                 0.9099
ret_ac                                                    0.0125
roll_win                 